# 09 — pandas for analysis

Notebook 07 was the nouns (Series, DataFrame). This is the verbs you will use every day as an analyst: clean, join, aggregate, reshape.

pandas is already installed in the `pro` env.

**Table of contents**

- [Missing values](#missing-values)
- [dtypes](#dtypes)
- [datetime](#datetime)
- [merge](#merge)
- [groupby](#groupby)
- [pivot and melt](#pivot-and-melt)
- [Try this](#try-this)

In [ ]:
import pandas as pd

sales = pd.DataFrame(
    {
        "order_id": [1, 2, 3, 4, 5],
        "city": ["Philly", "NYC", "Philly", None, "NYC"],
        "product": ["bike", "lock", "helmet", "bike", "lock"],
        "amount": [120, 25, 40, 120, None],
        "sold_on": ["2024-01-15", "2024-01-16", "2024-02-01", "2024-02-03", "2024-02-10"],
    }
)
customers = pd.DataFrame(
    {
        "city": ["Philly", "NYC", "Boston"],
        "region": ["Mid-Atlantic", "Northeast", "Northeast"],
    }
)
sales

## Missing values

In [ ]:
# None / NaN is "we do not know" — it is not zero
print(sales.isna().sum())
print(sales.dropna())              # drops any row with a missing value
print(sales.fillna({"city": "Unknown", "amount": 0}))

## dtypes

In [ ]:
# Wrong dtypes silently break math. Check them.
print(sales.dtypes)
sales["amount"] = pd.to_numeric(sales["amount"], errors="coerce")
print(sales["amount"].sum())  # 305.0  — NaN is skipped

## datetime

In [ ]:
sales["sold_on"] = pd.to_datetime(sales["sold_on"])
print(sales["sold_on"].dt.month)
print(sales["sold_on"].dt.to_period("M"))
sales[sales["sold_on"] >= "2024-02-01"]

## merge

In [ ]:
# merge = SQL JOIN. This is how you combine two tables.
merged = sales.merge(customers, on="city", how="left")
merged
# Boston has no sales, so it does not appear (left join keeps sales rows).
# city None will have region NaN.

## groupby

In [ ]:
# agg lets you compute several stats at once
(
    merged.groupby("region", dropna=False)["amount"]
    .agg(total="sum", orders="count", avg="mean")
    .reset_index()
)

## pivot and melt

In [ ]:
# pivot: long → wide. melt: wide → long.
wide = (
    sales.dropna(subset=["city"])
    .pivot_table(index="city", columns="product", values="amount", aggfunc="sum")
)
print(wide)

long = wide.reset_index().melt(id_vars="city", var_name="product", value_name="amount")
long

## Try this

1. Fill missing `city` with `"Unknown"` and missing `amount` with the column mean (not 0).
2. Inner-merge sales to customers. How many rows disappear, and why?
3. Make a pivot of **month** by **city** with `sum` of amount.